# CropDetector — Phase 9, Part 2
## Add real non-plant photographs and build a leakage-screened input-validation dataset

**Standalone Colab notebook.** It uses only your downloaded `CropDetector_Phase9_Audit.zip` (from Part 1). No old 130-cell runtime, classifier checkpoint, or Android files needed.

**Deliverable:** An auditable, training-ready dataset for two future validation stages:
- **Stage A — Plant-like vs Non-plant:** Treat *supported*, *unsupported crops*, and *unsupported tomato conditions* as **Plant-like**. A tomato leaf with an unsupported disease is not a non-plant image.
- **Stage B — Known crop vs other crop:** Maize, Grape, Potato, Tomato, and **Other Crop**. Tomato conditions that our disease classifier does not recognize still have crop label **Tomato**.

We use [PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset) (same audited commit) and [Natural Images](https://www.kaggle.com/datasets/prasunroy/natural-images). Natural Images contains airplane, car, cat, dog, flower, fruit, motorbike, and person; only **airplane, car, cat, dog, motorbike, person** are used as clear non-plant negatives. Flowers and fruits are intentionally **not** labelled non-plant. Their inclusion would blur the task. Natural Images is compiled from third-party sources; inspect rights before redistributing photographs.

**Critical limitations:** this is for initial development, *not* a definitive field evaluation. Cross-domain backgrounds and photography styles may be shortcuts. Existing PlantDoc test images were inspected in earlier phases and remain diagnostics only. We will need independent smartphone photographs, including phones, bottles, tools, blurred leaves, foliage with hands, and unseen crops.

## Cell 1 — Setup and reproducibility

In [ ]:
%pip -q install kagglehub imagehash

from pathlib import Path
from zipfile import ZipFile
from IPython.display import display
from PIL import Image, ImageOps, UnidentifiedImageError
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import imagehash, hashlib, subprocess, shutil, json, io, os, sys
from google.colab import files

SEED = 42
PHASH_THRESHOLD = 4
MAX_NONPLANT_PER_CLASS = 500
NONPLANT_FOLDERS = ('airplane', 'car', 'cat', 'dog', 'motorbike', 'person')
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
ROOT = Path('/content')
OUT = ROOT / 'CropDetector_Phase9_Part2'
OUT.mkdir(parents=True, exist_ok=True)
print('Part 2 initialized. Output:', OUT)

## Cell 2 — Upload the Part 1 audit ZIP and validate it

When prompted, upload **`CropDetector_Phase9_Audit.zip`** from your Mac. This file contains CSV manifests, not the photographs.

In [ ]:
AUDIT_ZIP = ROOT / 'CropDetector_Phase9_Audit.zip'
if not AUDIT_ZIP.exists():
    uploaded = files.upload()
    assert 'CropDetector_Phase9_Audit.zip' in uploaded, 'Upload the exact Phase 9 Part 1 audit ZIP.'

with ZipFile(AUDIT_ZIP) as z:
    expected_files = {'plantdoc_full_audit.csv','plantdoc_train.csv','plantdoc_dev.csv',
                      'plantdoc_legacy_test_diagnostic.csv','audit_report.json'}
    assert expected_files <= set(z.namelist()), 'Incomplete Part 1 audit ZIP.'
    audit_info = json.loads(z.read('audit_report.json'))
    pd_full = pd.read_csv(io.BytesIO(z.read('plantdoc_full_audit.csv')))
    pd_train = pd.read_csv(io.BytesIO(z.read('plantdoc_train.csv')))
    pd_dev = pd.read_csv(io.BytesIO(z.read('plantdoc_dev.csv')))
    pd_legacy = pd.read_csv(io.BytesIO(z.read('plantdoc_legacy_test_diagnostic.csv')))

assert len(pd_full)==audit_info['source_images']==2578
assert len(pd_train)==audit_info['train_images']==1684
assert len(pd_dev)==audit_info['development_images']==420
assert len(pd_legacy)==audit_info['legacy_test_diagnostic_images']==212
assert set(pd_train.Group).isdisjoint(pd_dev.Group)
assert set(pd_train.Group).isdisjoint(pd_legacy.Group)
assert set(pd_dev.Group).isdisjoint(pd_legacy.Group)
assert set(pd_train.SHA256).isdisjoint(pd_dev.SHA256)

print('PlantDoc commit:', audit_info['git_commit'])
print('Train:',len(pd_train),'Development:',len(pd_dev),'Legacy diagnostic:',len(pd_legacy))
display(pd.concat([pd_train.assign(Partition='Train'),pd_dev.assign(Partition='Development')])
        .groupby(['Partition','Role']).size().rename('Images').to_frame())

## Cell 3 — Recover the exact PlantDoc image files

In [ ]:
PLANTDOC_ROOT = ROOT / 'PlantDoc-Dataset'
COMMIT = audit_info['git_commit']
URL = audit_info['plantdoc_source']
if not (PLANTDOC_ROOT/'.git').exists():
    subprocess.run(['git','clone','--depth','1',URL,str(PLANTDOC_ROOT)],check=True)
head = subprocess.check_output(['git','-C',str(PLANTDOC_ROOT),'rev-parse','HEAD'],text=True).strip()
if head != COMMIT:
    subprocess.run(['git','-C',str(PLANTDOC_ROOT),'fetch','--depth','1','origin',COMMIT],check=True)
    subprocess.run(['git','-C',str(PLANTDOC_ROOT),'checkout','--detach',COMMIT],check=True)
head = subprocess.check_output(['git','-C',str(PLANTDOC_ROOT),'rev-parse','HEAD'],text=True).strip()
assert head == COMMIT, 'PlantDoc source version does not match audit.'

missing = [p for p in pd_full.RelativePath if not (PLANTDOC_ROOT/p).is_file()]
assert not missing, f'Missing images from pinned PlantDoc checkout: {missing[:5]}'
print('PlantDoc files verified:',len(pd_full))

## Cell 4 — Download the Natural Images dataset

Public dataset: [Prasun Roy, Natural Images](https://www.kaggle.com/datasets/prasunroy/natural-images). It contains **6,899 photographs across eight source categories**, but we will use six clearly non-plant categories. Download may require access to Kaggle; if KaggleHub reports an authentication/network error, open the dataset page and enable access rather than silently replacing it with unrelated data.

In [ ]:
NATURAL_ROOT = Path(kagglehub.dataset_download('prasunroy/natural-images'))
print('Natural Images downloaded to:', NATURAL_ROOT)
rows=[]
for p in sorted(NATURAL_ROOT.rglob('*')):
    if p.is_file() and p.suffix.lower() in IMG_EXT and p.parent.name.lower() in NONPLANT_FOLDERS:
        rows.append({'Source':'NaturalImages', 'ObjectClass':p.parent.name.lower(),
                     'RelativePath':p.relative_to(NATURAL_ROOT).as_posix(), 'Path':str(p)})
natural_all=pd.DataFrame(rows)
assert not natural_all.empty, ('Could not find expected class directories. '
                              f'Inspect {NATURAL_ROOT} and adjust discovery only after verifying structure.')
missing_classes=set(NONPLANT_FOLDERS)-set(natural_all.ObjectClass)
assert not missing_classes, f'Missing object classes: {missing_classes}'
print('Available non-plant photographs:',len(natural_all))
display(natural_all.groupby('ObjectClass').size().rename('Available').to_frame())

## Cell 5 — Deterministically sample up to 500 photographs per non-plant category

The cap keeps the experiment practical in free Colab and limits the class imbalance. We fingerprint all selected examples, then remove unreadable files and near-duplicates.

In [ ]:
selected=[]
for category, part in natural_all.groupby('ObjectClass',sort=True):
    part=part.sort_values('RelativePath')
    chosen=part.sample(n=min(MAX_NONPLANT_PER_CLASS,len(part)),random_state=SEED)
    selected.append(chosen)
nonplant=pd.concat(selected,ignore_index=True).sort_values(['ObjectClass','RelativePath']).reset_index(drop=True)

def fingerprint(path):
    data=Path(path).read_bytes()
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        width,height=im.size
        ph=int(str(imagehash.phash(im,hash_size=8)),16)
    return hashlib.sha256(data).hexdigest(), f'{ph:016x}',width,height

rows=[]
for r in nonplant.itertuples(index=False):
    try:
        sha,ph,w,h=fingerprint(r.Path)
        if min(w,h) < 64:
            continue  # avoid tiny thumbnails (a potential shortcut)
        rows.append({'Source':r.Source,'ObjectClass':r.ObjectClass,
                     'RelativePath':r.RelativePath,'SHA256':sha,
                     'pHash':ph,'Width':w,'Height':h})
    except (OSError,ValueError,UnidentifiedImageError) as exc:
        print('Skipping unreadable image:',r.RelativePath,type(exc).__name__)

nonplant=pd.DataFrame(rows)
assert not nonplant.empty
print('Valid selected non-plant images:',len(nonplant))
display(nonplant.groupby('ObjectClass').size().rename('Valid').to_frame())

## Cell 6 — Group Natural Images duplicates and check overlaps with PlantDoc

Group exact SHA-256 matches and pHash Hamming distance ≤4. If visually similar images are assigned different object-class labels, exclude the entire group. Also screen non-plant photos against **all** PlantDoc images, not just the PlantDoc train partition. pHash proximity is a conservative warning, not proof of identical provenance.

In [ ]:
class UnionFind:
    def __init__(self,n):
        self.p=list(range(n));self.rank=[0]*n
    def find(self,i):
        while self.p[i]!=i:
            self.p[i]=self.p[self.p[i]];i=self.p[i]
        return i

    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.rank[a]<self.rank[b]:a,b=b,a
        self.p[b]=a
        if self.rank[a]==self.rank[b]:self.rank[a]+=1

uf=UnionFind(len(nonplant))
hashes=[int(x,16) for x in nonplant.pHash]
first_sha={}
for i,sha in enumerate(nonplant.SHA256):
    if sha in first_sha:uf.union(i,first_sha[sha])
    else:first_sha[sha]=i

near_pairs=0
for i in range(len(nonplant)):
    for j in range(i+1,len(nonplant)):
        if (hashes[i]^hashes[j]).bit_count()<=PHASH_THRESHOLD:
            uf.union(i,j);near_pairs+=1

nonplant['Group']=[f'ni_{uf.find(i):05d}' for i in range(len(nonplant))]
conflicted=set(nonplant.groupby('Group').filter(lambda x:x.ObjectClass.nunique()>1).Group)

plant_hashes=[int(h,16) for h in pd_full.pHash.dropna()]
plant_shas=set(pd_full.SHA256.dropna())
plant_overlap=[]
for r in nonplant.itertuples():
    match=(r.SHA256 in plant_shas) or any((int(r.pHash,16)^h).bit_count()<=PHASH_THRESHOLD for h in plant_hashes)
    plant_overlap.append(match)
nonplant['NearPlantDoc']=plant_overlap
plant_overlap_groups=set(nonplant.loc[nonplant.NearPlantDoc,'Group'])

safe_nonplant=nonplant.loc[
    ~nonplant.Group.isin(conflicted|plant_overlap_groups)
].copy()
print('Natural Images pHash near pairs:',near_pairs)
print('Conflicting-category groups removed:',len(conflicted))
print('Groups matching PlantDoc removed:',len(plant_overlap_groups))
print('Photographs retained:',len(safe_nonplant),'/',len(nonplant))
assert len(safe_nonplant)>0

## Cell 7 — Make group-disjoint non-plant train/development partitions

All images in a near-duplicate group remain in one partition. Group stratification is performed per source category; no Natural Images test set is claimed as untouched final evaluation.

In [ ]:
rng=np.random.default_rng(SEED)
dev_ids=set()
for cls,part in safe_nonplant.groupby('ObjectClass',sort=True):
    groups=sorted(part.Group.unique())
    assert len(groups)>5, f'Too few groups for {cls}: {len(groups)}'
    ndev=max(1,round(0.20*len(groups)))
    dev_ids.update(rng.choice(groups,size=ndev,replace=False).tolist())

np_train=safe_nonplant.loc[~safe_nonplant.Group.isin(dev_ids)].copy()
np_dev=safe_nonplant.loc[safe_nonplant.Group.isin(dev_ids)].copy()
assert set(np_train.Group).isdisjoint(np_dev.Group)
assert set(np_train.SHA256).isdisjoint(np_dev.SHA256)

display(pd.concat([np_train.assign(Partition='Train'),np_dev.assign(Partition='Development')])
        .groupby(['Partition','ObjectClass']).size().rename('Images').to_frame())
print('Train non-plant:',len(np_train),'Development non-plant:',len(np_dev))

## Cell 8 — Assemble the training manifests for **two distinct tasks**

**Stage A:** `Target` = 0 for any PlantDoc plant image (including unsupported crops and tomato conditions), 1 for clear non-plant images.

**Stage B:** `CropLabel` = Maize / Grape / Potato / Tomato / Other Crop for PlantDoc **plant** rows. A tomato leaf with an unsupported disease remains `Tomato`. Do **not** train Stage B on Natural Images, because the crop class is not applicable to these non-plant photos.

In [ ]:
def crop_class(row):
    if row.Role == 'Unsupported Crop':return 'Other Crop'
    if row.Role == 'Unsupported Condition':return 'Tomato'
    i=int(row.Label)
    if 0<=i<=3:return 'Maize'
    if 4<=i<=7:return 'Grape'
    if 8<=i<=10:return 'Potato'
    if 11<=i<=14:return 'Tomato'
    raise ValueError(f'Cannot map crop: {row}')

def prepare_plant(df,partition):
    out=df[['RelativePath','SHA256','pHash','Group','Folder','Role','Label']].copy()
    out['Source']='PlantDoc'
    out['Partition']=partition
    out['Target']=0
    out['CropLabel']=df.apply(crop_class,axis=1)
    return out

def prepare_nonplant(df,partition):
    out=pd.DataFrame({
        'RelativePath':df.RelativePath,'SHA256':df.SHA256,
        'pHash':df.pHash,'Group':df.Group,'Folder':df.ObjectClass,
        'Role':'Non-plant','Label':-1,'Source':'NaturalImages',
        'Partition':partition,'Target':1,'CropLabel':''
    })
    return out

p_train=prepare_plant(pd_train,'train')
p_dev=prepare_plant(pd_dev,'dev')
train=pd.concat([p_train,prepare_nonplant(np_train,'train')],ignore_index=True)
dev=pd.concat([p_dev,prepare_nonplant(np_dev,'dev')],ignore_index=True)

assert set(train.Group).isdisjoint(dev.Group)
assert set(train.SHA256).isdisjoint(dev.SHA256)
assert not (train.Source.eq('PlantDoc') & train.CropLabel.isna()).any()
assert (train.loc[train.Role=='Unsupported Condition','CropLabel']=='Tomato').all()
assert (train.loc[train.Role=='Unsupported Crop','CropLabel']=='Other Crop').all()

print('Stage A train:',len(train),'dev:',len(dev))
display(pd.concat([train,dev]).groupby(['Partition','Source','Role']).size().rename('Images').to_frame())
print('Stage B crop train classes:',p_train.CropLabel.value_counts().to_dict())
print('Stage B crop dev classes:',p_dev.CropLabel.value_counts().to_dict())

## Cell 9 — Visual audit with sample images

Inspect for misleading natural-image labels: people holding leaves, trees filling backgrounds, plants dominating car photographs, etc. A quick grid is **not** a substitute for comprehensive annotation review.

In [ ]:
fig, axes=plt.subplots(2,6,figsize=(16,5))
for ax,(cls,part) in zip(axes[0],safe_nonplant.groupby('ObjectClass')):
    rr=part.sample(1,random_state=SEED).iloc[0]
    with Image.open(NATURAL_ROOT/rr.RelativePath) as im:
        ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
    ax.set_title('Non-plant: '+cls)
    ax.axis('off')
for ax,(role,part) in zip(axes[1],pd_train.groupby('Role')):
    rr=part.sample(1,random_state=SEED).iloc[0]
    with Image.open(PLANTDOC_ROOT/rr.RelativePath) as im:
        ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
    ax.set_title(role)
    ax.axis('off')
for ax in axes.flat:
    if not ax.has_data():ax.axis('off')
plt.suptitle('Spot-check source labels and visual domains')
plt.tight_layout()
plt.savefig(OUT/'dataset_visual_spotcheck.png',dpi=160)
plt.show()

## Cell 10 — Save portable manifests, audit summaries and download

This ZIP contains **manifests and reports, not the image datasets**. Keep both the Phase 9 Part 1 and Part 2 audit ZIPs on your Mac. Later notebooks will recreate the exact source paths and verify file hashes.

In [ ]:
for name,frame in [('validator_train.csv',train),('validator_dev.csv',dev),
                   ('nonplant_full_sample.csv',nonplant),('nonplant_safe.csv',safe_nonplant)]:
    frame.to_csv(OUT/name,index=False)

counts=(pd.concat([train,dev]).groupby(['Partition','Role']).size()
        .rename('Images').reset_index())
counts.to_csv(OUT/'split_counts.csv',index=False)

report={
    'phase':'Phase9_Part2',
    'seed':SEED,
    'plantdoc_git_commit':COMMIT,
    'plantdoc_audit_zip_sha256':hashlib.sha256(AUDIT_ZIP.read_bytes()).hexdigest(),
    'natural_images_source':'https://www.kaggle.com/datasets/prasunroy/natural-images',
    'natural_images_local_cache_location':str(NATURAL_ROOT),
    'nonplant_categories':list(NONPLANT_FOLDERS),
    'excluded_ambiguous_natural_categories':['flower','fruit'],
    'nonplant_per_category_cap':MAX_NONPLANT_PER_CLASS,
    'phash_distance_threshold':PHASH_THRESHOLD,
    'nonplant_sampled':len(nonplant),
    'nonplant_plantdoc_overlap_group_count':len(plant_overlap_groups),
    'nonplant_conflicting_label_groups':len(conflicted),
    'nonplant_safe_count':len(safe_nonplant),
    'stage_a_train':len(train),'stage_a_dev':len(dev),
    'plant_train':len(p_train),'plant_dev':len(p_dev),
    'nonplant_train':len(np_train),'nonplant_dev':len(np_dev),
    'stage_a_labels':{'Plant-like':0,'Non-plant':1},
    'stage_b_crop_labels':['Maize','Grape','Potato','Tomato','Other Crop'],
    'train_dev_group_overlap':len(set(train.Group)&set(dev.Group)),
    'train_dev_sha_overlap':len(set(train.SHA256)&set(dev.SHA256)),
    'legacy_plantdoc_test_used_for_training':False,
    'final_independent_field_test_available':False,
    'notes':[
        'Unsupported tomato conditions are valid Tomato crop images, not non-plants.',
        'Natural Images object photographs are not representative of all irrelevant camera inputs.',
        'Never treat this development split as an untouched final field benchmark.',
        'Source photographs are not included in the exported ZIP.'
    ]
}
(OUT/'audit_report.json').write_text(json.dumps(report,indent=2))
(OUT/'README.txt').write_text(
  'CropDetector Phase 9 Part 2: non-plant dataset acquisition and validator manifests.\n'
  'To restore images: clone PlantDoc at the pinned commit and re-download Kaggle Natural Images.\n'
  'RelativePath in each CSV is resolved against the source root specified by Source.\n'
  'No original image files are redistributed in this ZIP.\n'
)
archive=shutil.make_archive(str(ROOT/'CropDetector_Phase9_Part2_Data'),'zip',OUT)
print(json.dumps(report,indent=2))
print('Archive:',archive)
files.download(archive)

## Next — Part 3: Train a lightweight Plant-like / Non-plant validator

**Part 3** will use these exported manifests to train a small MobileNetV3Small or MobileNetV2-family binary gate with class-balanced sampling, evaluate per-object-category error rates (not just overall accuracy), and separately prototype known-crop recognition. We will compare against an independent camera test set containing tools, bottles, phones, leaves in clutter, unsupported crops, and low-quality images. Only then will we export INT8 and measure **combined** validation + disease-classification latency on your Exynos 1330.

**Send next:** the generated `CropDetector_Phase9_Part2_Data.zip`, and any error output if KaggleHub cannot download the public dataset.